In [ ]:
from pathlib import Path

base_path = Path("in")
input_file = base_path / "Ricevute.pdf"

print(input_file)
print(input_file.name)
print(input_file.stem)
print(input_file.suffix)


In [ ]:
cartella_corrente = Path(".")
cartella_input = Path(cartella_corrente / "in")

# Cerca solo i PDF nella cartella corrente
percorsi_pdf = list(cartella_input.glob("*.pdf"))
print(f"Trovati {len(percorsi_pdf)} file PDF")

# Cerca PDF, JPG e PNG in modo ricorsivo
estensioni = ("*.pdf", "*.jpg", "*.jpeg", "*.png")
tutti_i_files = []
for ext in estensioni:
    tutti_i_files.extend(cartella_corrente.rglob(ext))

print(f"Trovati {len(tutti_i_files)} documenti.")


In [ ]:
import fitz # PyMuPDF
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pymupdf import Pixmap

In [ ]:
import os
os.makedirs("out", exist_ok=True)

In [ ]:
# Estrazione testo rapida (se il PDF non è una scansione)

doc = fitz.open("in/git-slides.pdf")
print(f"git-slides: Pagine totali: {len(doc)}")
print(f"Metadati: {doc.metadata}")

for pagina in doc:
    testo = pagina.get_text(sort=False)
    print(testo)


In [ ]:
# Convertire PDF in immagini
doc = fitz.open("in/Ricevute.pdf")
pagina = doc[0] # Prendi la prima pagina

# Aumenta la risoluzione (2.0 = 144 DPI, 4.0 = 288 DPI)
for zoom in [2, 4, 8, 16]:
    mat = fitz.Matrix(72 * zoom / 72, 72 * zoom / 72)
    clip=None
    clip=(15,20,160,50)
    pix = pagina.get_pixmap(matrix=mat, clip=clip)

    # Salvataggio per il prossimo step (Preprocessing)
    pix.save(f"out/Ricevute_zoom{zoom}.png")
    img = np.ndarray([pix.h, pix.w, 3], dtype=np.uint8, buffer=pix.samples_mv)
    plt.figure()
    plt.title(f'zoom {zoom}x -> {72*zoom}DPI')
    plt.imshow(img)

In [ ]:
# Lettura libretto moto
def gray_scaleup(nome, immagine, zoom):
    new_size = [d * zoom for d in immagine.size]
    immagine_grigia = immagine.convert("L")
    immagine_grigia_scalata = immagine_grigia.resize(new_size)
    immagine_grigia_scalata.save(f"out/{nome}_zoom_{zoom}_grigia_scalata.png")
    return immagine_grigia_scalata

img = Image.open("in/LibrettoMotoRitaglio.jpeg")
dimensioni_selezione = (0, 54, 535, 54 + 225)
print(img.size)  # (Larghezza, Altezza)
print(img.format)  # PNG
img = img.crop(dimensioni_selezione)
display(img)

print('gray e scaleup')
for zoom in [2, 16]:
    display(gray_scaleup("LibrettoMoto", img, zoom))


In [ ]:
# Pieghevole Lanterna

def convert_fitz_to_pil(immagine_pixmap: Pixmap) -> Image.Image:
    immagine = Image.frombytes("RGB", (immagine_pixmap.width, immagine_pixmap.height), immagine_pixmap.samples)
    return immagine

def convert_colors_from_pil_to_cv2(image):
    return cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)

def convert_colors_from_cv2_to_pil(image):
    return Image.fromarray(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))

def lanterna_colori_pdf():
    coord_riquadro_100dpi = (24, 139, 364, 326)
    # for dpi_orig in [100, 200, 300, 600, 1200]:
    for dpi_orig in [100, 300, 600]:
        nome_immagine = f"Lanterna-{dpi_orig}dpi"
        fitz_documento = fitz.open(f"in/{nome_immagine}.pdf")
        immagine_pixmap = fitz_documento[0].get_pixmap(dpi=dpi_orig)
        # convert to Image
        immagine = convert_fitz_to_pil(immagine_pixmap)
        immagine = immagine.rotate(90, expand=True)
        immagine.save(f"out/{nome_immagine}_ruotata.png")
        coord_riquadro = tuple( c * dpi_orig / 100 for c in coord_riquadro_100dpi)
        print("Coordinate riquadro:", dpi_orig, coord_riquadro)
        immagine = immagine.crop(coord_riquadro)
        immagine.save(f"out/{nome_immagine}_ruotata_tagliata.png")

        display(immagine)
        
        # for zoom in [2, 4, 8, 16]:
        for zoom in [2, 4]:
            print("Imagine DPI:", dpi_orig, 'Zoom:', zoom)
            immagine_grigia_scalata = gray_scaleup(nome_immagine, immagine, zoom)
            # show image
            # plt.imshow(immagine_grigia_scalata)
            display(immagine_grigia_scalata)

# try_with_original_pdf()
# convert_scanned_pdf()
# first_try_with_converted_pdf()
# libretto_moto_colori_pdf()
lanterna_colori_pdf()


In [ ]:
def ridimensiona_proporzionale(image_path, max_width=2000):
    with Image.open(image_path) as img:
        w_percent = (max_width / float(img.size[0]))
        h_size = int((float(img.size[1]) * float(w_percent)))
        img = img.resize((max_width, h_size))
        img.save("out/Lanterna-1200dpi_ottimizzata.png", "PNG", quality=95)
        display(img)

ridimensiona_proporzionale("out/Lanterna-1200dpi_ruotata.png")
